# 제목 
Learning Face - Mini Study Coach with LangGraph

#그래프 구조 
START → analyze_request → make_study_plan → make_quiz → END

In [1]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

In [2]:
# State 정의
class StudyState(TypedDict):
    user_topic: str
    level: str
    plan: List[str]
    quiz: List[str]

In [3]:
# Node 1: 사용자 요청 분석
def analyze_request(state: StudyState):
    topic = state["user_topic"].lower()

    if "초보" in topic or "beginner" in topic:
        level = "beginner"
    elif "중급" in topic or "intermediate" in topic:
        level = "intermediate"
    else:
        level = "beginner"

    return {
        "level": level
    }

In [4]:
# Node 2: 학습 계획 생성
def make_study_plan(state: StudyState):
    topic = state["user_topic"]
    level = state["level"]

    if level == "beginner":
        plan = [
            f"{topic}의 핵심 개념 3가지를 이해한다.",
            f"{topic}의 쉬운 예제를 2개 본다.",
            f"{topic} 관련 짧은 연습문제 1개를 풀어본다."
        ]
    else:
        plan = [
            f"{topic}의 핵심 개념을 빠르게 복습한다.",
            f"{topic}의 실전 예제를 분석한다.",
            f"{topic}를 활용한 응용 문제를 풀어본다."
        ]

    return {
        "plan": plan
    }

In [5]:
# Node 3: 복습 질문 생성
def make_quiz(state: StudyState):
    topic = state["user_topic"]

    quiz = [
        f"{topic}에서 가장 중요한 개념은 무엇인가요?",
        f"{topic}를 실제로 사용할 수 있는 예시는 무엇인가요?"
    ]

    return {
        "quiz": quiz
    }

In [6]:
# 그래프 만들기
builder = StateGraph(StudyState)

builder.add_node("analyze_request", analyze_request)
builder.add_node("make_study_plan", make_study_plan)
builder.add_node("make_quiz", make_quiz)

builder.add_edge(START, "analyze_request")
builder.add_edge("analyze_request", "make_study_plan")
builder.add_edge("make_study_plan", "make_quiz")
builder.add_edge("make_quiz", END)

graph = builder.compile()

In [7]:
# 실행 예시
result = graph.invoke({
    "user_topic": "파이썬 if문 초보",
    "level": "",
    "plan": [],
    "quiz": []
})

result

{'user_topic': '파이썬 if문 초보',
 'level': 'beginner',
 'plan': ['파이썬 if문 초보의 핵심 개념 3가지를 이해한다.',
  '파이썬 if문 초보의 쉬운 예제를 2개 본다.',
  '파이썬 if문 초보 관련 짧은 연습문제 1개를 풀어본다.'],
 'quiz': ['파이썬 if문 초보에서 가장 중요한 개념은 무엇인가요?',
  '파이썬 if문 초보를 실제로 사용할 수 있는 예시는 무엇인가요?']}

In [8]:
print("학습 수준:", result["level"])
print("\n학습 계획:")
for step in result["plan"]:
    print("-", step)

print("\n복습 질문:")
for q in result["quiz"]:
    print("-", q)

학습 수준: beginner

학습 계획:
- 파이썬 if문 초보의 핵심 개념 3가지를 이해한다.
- 파이썬 if문 초보의 쉬운 예제를 2개 본다.
- 파이썬 if문 초보 관련 짧은 연습문제 1개를 풀어본다.

복습 질문:
- 파이썬 if문 초보에서 가장 중요한 개념은 무엇인가요?
- 파이썬 if문 초보를 실제로 사용할 수 있는 예시는 무엇인가요?
